# Sila na zakrivljenu plohu — četvrtina kruga

**Poglavlje 6: Zakrivljene plohe i rastav sila**

Ovaj interaktivni prikaz nadopunjuje izvod hidrostatičke sile na zakrivljenu plohu kroz primjer četvrtine kruga uronjene u vodu. Mijenjanjem polumjera krivulje, dubine vrha i širine plohe prati se rastav sile na horizontalnu i vertikalnu komponentu.

## Cilj

Na zakrivljenoj plohi hidrostatička sila razlaže se na horizontalnu komponentu (jednaku sili na vertikalnu projekciju plohe) i vertikalnu komponentu (jednaku težini imaginarnog volumena fluida iznad ili ispod plohe). Prikaz omogućuje:

1. mijenjanje polumjera plohe $R$;
2. mijenjanje dubine vrha plohe $h_t$;
3. mijenjanje širine plohe $L$;
4. praćenje komponenti $F_H$ i $F_V$ te rezultante $F_R$.

## Pretpostavke modela

- jednolika gustoća vode, $\rho = 998$ kg/m³;
- ploha je četvrtina kruga polumjera $R$ s vrhom na dubini $h_t$;
- konveksna strana plohe okrenuta je prema fluidu;
- statičko stanje, bez strujanja.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## Računski model

Horizontalna komponenta sile jednaka je sili na vertikalnu projekciju plohe — pravokutnik visine $R$ i širine $L$:

$$F_H = \rho g L R \left(h_t + \frac{R}{2}\right).$$

Vertikalna komponenta jednaka je težini imaginarnog volumena fluida iznad zakrivljene plohe:

$$F_V = \rho g L \left(h_t R + R^2 - \frac{\pi R^2}{4}\right).$$

Iznos rezultante i kut prema horizontali:

$$F_R = \sqrt{F_H^2 + F_V^2}, \qquad \tan\varphi = \frac{F_V}{F_H}.$$

In [ ]:
RHO = 998.0
G = 9.81

def zakrivljena(R, h_t, L):
    F_H = RHO * G * L * R * (h_t + R/2)
    V_imag = L * (h_t * R + R**2 - np.pi * R**2 / 4)
    F_V = RHO * G * V_imag
    F_R = np.sqrt(F_H**2 + F_V**2)
    phi = np.degrees(np.arctan2(F_V, F_H))
    return {'F_H': F_H, 'F_V': F_V, 'F_R': F_R, 'phi': phi}

## Interaktivni prikaz

Klizačima u nastavku biraju se polumjer, dubina vrha i širina plohe. Prikaz pokazuje bočni presjek četvrtine kruga uronjene u vodu s vektorima komponenti sile.

In [ ]:
def zakrivljena_prikaz(R, h_t, L):
    r = zakrivljena(R, h_t, L)

    fig, ax = plt.subplots(figsize=(8, 6))

    # Slobodna površina
    ax.axhline(0, color='#1565c0', lw=2)
    ax.fill_between([-2, 2], 0, -h_t - 1.5*R,
                     fc='#aed6f1', alpha=0.3)

    # Četvrtina kruga (vrh u (0, -h_t), središte kruga u (R, -h_t))
    theta = np.linspace(np.pi, 1.5*np.pi, 60)
    x_arc = R + R * np.cos(theta)
    y_arc = -h_t + R * np.sin(theta)
    ax.plot(x_arc, y_arc, color='#c62828', lw=4, label='ploha')
    ax.fill_between(x_arc, y_arc, -h_t,
                     fc='#f4cccc', alpha=0.3,
                     label='imaginarni volumen')

    # Hvatište — približno u težištu četvrtine kruga
    x_h = R - 4*R/(3*np.pi)
    y_h = -h_t - 4*R/(3*np.pi)

    # Vektor F_H
    skala = 0.001 / max(r['F_H'], r['F_V']) * R
    LH = r['F_H'] * skala
    LV = r['F_V'] * skala
    ax.annotate('', xy=(x_h - LH, y_h), xytext=(x_h, y_h),
                 arrowprops=dict(arrowstyle='->',
                                   color='#1976d2', lw=2.2))
    ax.text(x_h - LH/2, y_h + 0.05,
             f'$F_H$ = {r["F_H"]/1000:.1f} kN',
             color='#1976d2', fontsize=10, ha='center')

    # Vektor F_V
    ax.annotate('', xy=(x_h, y_h - LV), xytext=(x_h, y_h),
                 arrowprops=dict(arrowstyle='->',
                                   color='#2e7d32', lw=2.2))
    ax.text(x_h - 0.05, y_h - LV/2,
             f'$F_V$ = {r["F_V"]/1000:.1f} kN',
             color='#2e7d32', fontsize=10, ha='right')

    # Vektor rezultante
    ax.annotate('', xy=(x_h - LH, y_h - LV), xytext=(x_h, y_h),
                 arrowprops=dict(arrowstyle='->',
                                   color='#c62828', lw=2.5))
    ax.text(x_h - LH - 0.05, y_h - LV - 0.05,
             f'$F_R$ = {r["F_R"]/1000:.1f} kN\n'
             f'$\\varphi$ = {r["phi"]:.1f}°',
             color='#c62828', fontsize=10, ha='right')

    ax.set_xlim(-0.5, R*1.5 + 0.5)
    ax.set_ylim(-(h_t + R + 0.3), 0.3)
    ax.set_aspect('equal')
    ax.set_xlabel('horizontalna koordinata (m)')
    ax.set_ylabel('dubina (m)')
    ax.set_title(
        f'$R$ = {R:.2f} m,  $h_t$ = {h_t:.2f} m,  $L$ = {L:.2f} m'
    )
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(ls=':', alpha=0.5)

    plt.tight_layout()
    plt.show()


interact(
    zakrivljena_prikaz,
    R=FloatSlider(min=0.2, max=2.0, step=0.1, value=0.8,
                   description='$R$ (m)',
                   layout=Layout(width='420px')),
    h_t=FloatSlider(min=0, max=5, step=0.1, value=1.0,
                     description='$h_t$ (m)',
                     layout=Layout(width='420px')),
    L=FloatSlider(min=0.5, max=5.0, step=0.1, value=2.0,
                   description='$L$ (m)',
                   layout=Layout(width='420px'))
);

## Pitanja za istraživanje

1. **Granični slučaj plitke vode.** Što se događa s omjerom $F_V/F_H$ kada $h_t \to 0$? Zašto vertikalna komponenta postaje značajnija u plitkoj vodi?

2. **Granični slučaj duboke vode.** Za $h_t \gg R$, što vrijedi za omjer $F_V/F_H$? Postoji li gornja granica?

3. **Smjer rezultante.** Pri kojoj kombinaciji $R$ i $h_t$ rezultanta sile prolazi kroz središte krivulje? Što tehnički znači ta razdvojnica?

4. **Inženjerska primjena.** Brodska vrata u doku imaju zakrivljenu donju polovicu polumjera $R \approx 1$ m, vrh na dubini $h_t \approx 4$ m i širinu $L = 8$ m. Kolika je rezultanta sile po vratima i pod kojim kutem djeluje?

## Veza s teorijom poglavlja

Ovaj prikaz materijalizira ključno opažanje iz poglavlja 6: sila na zakrivljenu plohu rastavlja se na horizontalnu komponentu (jednaku sili na vertikalnu projekciju) i vertikalnu komponentu (jednaku težini imaginarnog volumena fluida iznad plohe). Taj rastav vrijedi neovisno o obliku krivulje sve dok se geometrija plohe može pratiti zatvorenim volumenom.